# Hospedando um Agente CrewAI Simples com modelos Amazon Bedrock no Amazon Bedrock AgentCore Runtime

## Visão Geral

Neste tutorial, aprenderemos como hospedar um agente CrewAI simples usando o Amazon Bedrock AgentCore Runtime. Utilizaremos a instrumentação Opentelemetry e a biblioteca Python AWS Opentelemetry para adicionar observabilidade a este agente e monitorar seu desempenho no Amazon CloudWatch GenAI Observability Dashboard.

### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                   |
| Tipo de agente      | Único                                                                            |
| Framework agêntico  | CrewAI                                                                           |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                     |
| Componentes do tutorial | Hospedagem de agente no AgentCore Runtime. Usando CrewAI e Amazon Bedrock Model |
| Vertical do tutorial | Cross-vertical                                                                  |
| Complexidade do exemplo | Fácil                                                                        |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK e boto3                                      |

### Funcionalidades Principais do Tutorial

* Hospedagem de Agentes no Amazon Bedrock AgentCore Runtime
* Uso de modelos Amazon Bedrock
* Uso do CrewAI
* Amazon CloudWatch GenAI Observability


### Arquitetura do Tutorial

Neste tutorial, descreveremos como implantar uma crew multi-agente existente no AgentCore Runtime. 

Para fins de demonstração, usaremos uma crew CrewAI utilizando modelos Amazon Bedrock

No nosso exemplo, usaremos um agente de viagens com capacidades de busca na web.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Credenciais AWS com as permissões apropriadas
* Amazon Bedrock AgentCore SDK
* CrewAI
* Acesso ao Amazon CloudWatch
* Habilitar [transaction search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) no Amazon CloudWatch.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Preparando seu agente para implantação no AgentCore Runtime

Vamos agora implantar nosso agente no AgentCore Runtime. Para isso, precisamos:
* Importar o Runtime App com `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Inicializar o App no nosso código com `app = BedrockAgentCoreApp()`
* Decorar a função de invocação com o decorator `@app.entrypoint`
* Deixar o AgentCoreRuntime controlar a execução do agente com `app.run()`

### Agente CrewAI com modelo Amazon Bedrock
Vamos criar nosso Agente CrewAI pronto para runtime usando o modelo Amazon Bedrock.

In [ ]:
%%writefile crewai_runtime_agent.py
import os

from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from ddgs import DDGS
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from opentelemetry.instrumentation.crewai import CrewAIInstrumentor

# Instrument CrewAI with Opentelemetry
# Note: The AWS OpenTelemetry distro will automatically handle tracer provider setup
# when using opentelemetry-instrument command
CrewAIInstrumentor().instrument()

app = BedrockAgentCoreApp()

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for current information about travel destinations, attractions, and events."""
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=3)
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )
        
        return "\n".join(formatted_results) if formatted_results else "No results found."
        
    except Exception as e:
        return f"Search error: {str(e)}"

def get_llm():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    
    try:
        llm = LLM(
            model=f"bedrock/{model_id}",
            temperature=0.7,
            max_tokens=512,
            aws_region_name=region
        )
        logger.info(f"Successfully initialized Bedrock LLM with model: {model_id} in region: {region}")
        return llm
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock LLM: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

@app.entrypoint
def crewai_agent_bedrock(payload, context):
    """
    Invoke the agent with a payload
    """
    print(f'Payload: {payload}')
    try:
        user_input = payload.get("prompt", "What are some interesting places to visit?")
        print(f"Processing request: {user_input}")
        
        llm = get_llm()

        travel_agent = Agent(
            role='Travel Destination Researcher',
            goal='Find dream destinations matching user preferences using web search for current information',
            backstory="You are an experienced travel agent specializing in personalized travel recommendations with access to real-time web information.",
            verbose=True,
            allow_delegation=False,
            llm=llm,
            max_iter=3,
            tools=[web_search]
        )

        task = Task(
            description=f"Research and provide travel recommendations based on this request: {user_input}. Use web search to find current information about venues, events, and attractions.",
            expected_output="A comprehensive list of recommended destinations with current information, brief descriptions, and practical travel details.",
            agent=travel_agent
        )

        crew = Crew(
            agents=[travel_agent],
            tasks=[task],
            verbose=True
        )

        result = crew.kickoff()
        
        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        return {"result": result.raw}
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return {"error": f"An error occurred: {str(e)}"}

if __name__ == "__main__":
    app.run()

## O que acontece nos bastidores?

Quando você usa o `BedrockAgentCoreApp`, ele automaticamente:

* Cria um servidor HTTP que escuta na porta 8080
* Implementa o endpoint `/invocations` necessário para processar os requisitos do agente
* Implementa o endpoint `/ping` para health checks (muito importante para agentes assíncronos)
* Gerencia os tipos de conteúdo e formatos de resposta adequados
* Gerencia o tratamento de erros de acordo com os padrões AWS

## Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções de configuração abrangentes, permitindo especificar imagens de container, variáveis de ambiente e configurações de criptografia. Você também pode configurar as definições de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente. 

**Nota:** A melhor prática operacional é empacotar o código como container e enviar para o ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCore Python SDK para empacotar facilmente seus artefatos e implantá-los no AgentCore Runtime.

### Configurar a implantação do AgentCore Runtime

A seguir, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu docker file será gerado com base no código da sua aplicação. 

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

Observe que ao usar o `bedrock_agentcore_starter_toolkit` para configurar seu agente, ele cuida da instrumentação opentelemetry. 

Ao configurar para ambiente containerizado (como docker), adicione o seguinte comando, um exemplo é fornecido abaixo:

`CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]`

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "simple_crewai_travel_agent"
response = agentcore_runtime.configure(
    entrypoint="crewai_runtime_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY'
)
response

### Lançando o agente no AgentCore Runtime

Agora que temos um docker file, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
# Disable CrewAI's built-in telemetry to avoid conflicts
launch_result = agentcore_runtime.launch(env_vars={
        "CREWAI_DISABLE_TELEMETRY": "true",
        "OTEL_PYTHON_EXCLUDED_URLS": "https://api.scarf.sh/"
    })
launch_result

### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar o status da implantação

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando o AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What are some cowboy-themed attractions and museums in Texas?"})
invoke_response

### Processando os resultados da invocação

Agora podemos processar os resultados da nossa invocação para incluí-los em uma aplicação

In [ ]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Invocando o AgentCore Runtime com boto3

Agora que seu AgentCore Runtime foi criado, você pode invocá-lo com qualquer AWS SDK. Por exemplo, você pode usar o método `invoke_agent_runtime` do boto3 para isso.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What are some rodeo events happening in Oklahoma?"})
)

response_body = boto3_response['response'].read()
response_data = json.loads(response_body)
display(Markdown(response_data.get('result', 'No result found')))

### Observabilidade do AgentCore no Amazon CloudWatch 

Para resumir, siga os passos abaixo para habilitar a observabilidade dos agentes hospedados no AgentCore Runtime: 

- Habilitar o Transaction Search no Amazon CloudWatch 
- O agente emite traces e é instrumentado usando o comando opentelemetry: `opentelemetry-instrument python any_runtime_agent.py`
- O arquivo requirements.txt contém `aws-opentelemetry-distro` listado ao implantar o agente no Bedrock AgentCore Runtime.

## Visão Geral do Bedrock AgentCore no GenAI Observability Dashboard 

Você pode visualizar todos os seus Agentes que possuem observabilidade e filtrar os dados com base em períodos de tempo.

No dashboard principal, você pode visualizar as métricas de runtime de todos os agentes.

Agora, se você clicar no agente que acabou de implantar, será direcionado a um dashboard com as métricas de runtime específicas deste agente, e também pode filtrar os dados por um período de tempo personalizado.

Na aba Sessions View, você pode navegar por todas as sessões associadas a este agente.



Na aba Trace View, você pode examinar as informações de traces e spans deste agente em runtime.


<div style="text-align:left">
    <img src="images/span_crew_Ai.png" width="60%"/>
</div>


Clique nas diversas funcionalidades do GenAI Observability Dashboard para obter informações mais detalhadas sobre os traces.

<div style="text-align:left">
    <img src="images/span_details.png" width="60%"/>
</div>




## Limpeza (Opcional)

Vamos agora limpar o AgentCore Runtime criado

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Parabéns!

Você criou e implantou com sucesso um agente CrewAI simples no Amazon Bedrock AgentCore Runtime com observabilidade habilitada. Este exemplo demonstra como:

- Criar um agente de viagens CrewAI simples com capacidades de busca na web
- Habilitar observabilidade através do Amazon CloudWatch
- Invocar o agente usando tanto o SDK quanto o boto3

O agente agora pode ser utilizado com capacidades completas de observabilidade e monitoramento.